In [ ]:
TOUR_NAME = "Almost All Village Halls Tour 2026"

In [7]:
import requests
import xml.etree.ElementTree as ET
import json
import re
from datetime import datetime
from urllib.parse import urlparse


def slugify(text):
    """Create a URL-safe slug"""
    text = text.lower()
    text = re.sub(r"[^\w\s-]", "", text)
    text = re.sub(r"\s+", "-", text)
    return text.strip("-")


def parse_date(date_str):
    """Convert 'Sat, 14 Feb 2026' → '14/02/2026'"""
    dt = datetime.strptime(date_str.strip(), "%a, %d %b %Y")
    return dt.strftime("%d/%m/%Y")


def parse_time(time_str):
    """Normalise times like '7:30pm' → '7:30PM'"""
    return time_str.strip().upper()


def extract_city_postcode(venue_text):
    """
    Example:
    'Northampton Welford Community Centre, NN6 6HJ'
    """
    parts = [p.strip() for p in venue_text.split(",")]
    postcode = parts[-1] if len(parts) > 1 else ""
    city = parts[-2] if len(parts) > 2 else ""
    return city, postcode


def fetch_and_convert(xml_url):
    response = requests.get(xml_url)
    response.raise_for_status()

    root = ET.fromstring(response.content)

    venues = {}
    events = []

    for event in root.findall("event"):

        support_el = event.find("support")
        support_text = (support_el.text or "").strip()

        title_text = (event.find("title").text or "")

        if TOUR_NAME not in support_text and "Gaz Brookfield" not in title_text:
            continue

        # --- VENUE ---
        venue_el = event.find("venue")
        venue_name_full = venue_el.text.strip()
        city, postcode = extract_city_postcode(venue_name_full)

        venue_name = venue_name_full.split(",")[0]
        venue_slug = slugify(venue_name)

        if venue_slug not in venues:
            venues[venue_slug] = {
                "id": venue_slug,
                "name": venue_name,
                "city": city,
                "postcode": postcode,
                "full_address": venue_name_full,
                "url": "",  # not in XML, leave blank or enrich later
            }

        # --- EVENT ---
        date_start = event.find("date/start").text
        time_info = event.find("time/info")

        event_data = {
            "date": parse_date(date_start),
            "time": parse_time(time_info.text) if time_info is not None else "",
            "price": "",
            "ticket_url": event.find("link").text.strip(),
            "isMusic": "music" in (event.find("genre").text or "").lower(),
            "venue_id": venue_slug,
        }

        events.append(event_data)

    return venues, events


XML_FEED_URL = "https://services.wegottickets.com/feeds/events.php?fid=328"

venues, events = fetch_and_convert(XML_FEED_URL)

output = {"venues": venues, "events": events}

print(json.dumps(output, indent=2))

{
  "venues": {
    "northampton-welford-community-centre": {
      "id": "northampton-welford-community-centre",
      "name": "Northampton Welford Community Centre",
      "city": "",
      "postcode": "NN6 6HJ",
      "full_address": "Northampton Welford Community Centre, NN6 6HJ",
      "url": ""
    },
    "ashby-de-la-zouch-packginton-village-hall": {
      "id": "ashby-de-la-zouch-packginton-village-hall",
      "name": "Ashby-de-la-Zouch Packginton Village Hall",
      "city": "",
      "postcode": "LE65 1WJ",
      "full_address": "Ashby-de-la-Zouch Packginton Village Hall, LE65 1WJ",
      "url": ""
    },
    "kingswood-village-hall": {
      "id": "kingswood-village-hall",
      "name": "Kingswood Village Hall",
      "city": "",
      "postcode": "GL12 8RF",
      "full_address": "Kingswood Village Hall, GL12 8RF",
      "url": ""
    },
    "matlock-the-burton-institute": {
      "id": "matlock-the-burton-institute",
      "name": "Matlock The Burton Institute",
      "ci